https://judge.nitro-ai.org/competitions/roai-2025/simulare-ojia/2/view?participant_name=ecaterina

# setup + imports

In [ ]:
import numpy as np
import pandas as pd
import math

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold

In [ ]:
import calendar

In [ ]:
train_data = pd.read_csv('/content/train_data.csv')
test_data = pd.read_csv('/content/test_data.csv')

y = train_data.Credit_Score
X = train_data.drop(['Credit_Score'], axis=1)

#first subtasks

In [ ]:
answers1 = len(train_data['ID'])

subtask1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": 1,
    "answer": answers1
}, index=[0])

In [ ]:
idx = []
for i in range(len(train_data)):
  if train_data['Credit_Utilization_Ratio'].iloc[i] >= 25:
    idx.append(i)

selectsalary = train_data['Monthly_Inhand_Salary'].iloc[idx]
answers2 = math.floor(selectsalary.mean())

subtask2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": 1,
    "answer": answers2
}, index=[0])

In [ ]:
answers3 = train_data['Month'].nunique()

subtask3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": 1,
    "answer": answers3
}, index=[0])

In [ ]:
idx = []
for i in range(len(train_data)):
  ssn = str(train_data['SSN'].iloc[i])
  last2 = ssn[-2:]
  if last2=='20':
    idx.append(i)

selectssn = train_data['SSN'].iloc[idx]
answers4 = selectssn.nunique()

subtask4 = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": 1,
    "answer": answers4
}, index=[0])

#data exploration

In [ ]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35879 entries, 0 to 35878
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        35879 non-null  object 
 1   Customer_ID               35879 non-null  object 
 2   Month                     35879 non-null  object 
 3   Name                      35879 non-null  object 
 4   Age                       35879 non-null  object 
 5   SSN                       35879 non-null  object 
 6   Occupation                35879 non-null  object 
 7   Annual_Income             35879 non-null  object 
 8   Monthly_Inhand_Salary     35879 non-null  float64
 9   Num_Bank_Accounts         35879 non-null  int64  
 10  Num_Credit_Card           35879 non-null  int64  
 11  Interest_Rate             35878 non-null  float64
 12  Num_of_Loan               35878 non-null  object 
 13  Type_of_Loan              35878 non-null  object 
 14  Delay_

In [ ]:
print(train_data.isna().sum().sum(), test_data.isna().sum().sum())
print("duplicate rows:", train_data.duplicated().sum())
train_data.nunique()

17 0
duplicate rows: 0


,0
ID,35879
Customer_ID,10913
Month,8
Name,9059
Age,758
SSN,10850
Occupation,16
Annual_Income,13377
Monthly_Inhand_Salary,11319
Num_Bank_Accounts,416


In [ ]:
train_data['Occupation'].nunique()

occount = {}
for e in train_data['Occupation']:
  for o in e.split(','):
    if o not in occount:
      occount[o] = 1
    else:
      occount[o] += 1

occount

{'Accountant': 2213,
 '_______': 2547,
 'Lawyer': 2370,
 'Journalist': 2080,
 'Musician': 2162,
 'Media_Manager': 2262,
 'Engineer': 2275,
 'Developer': 2245,
 'Scientist': 2217,
 'Architect': 2224,
 'Teacher': 2267,
 'Doctor': 2260,
 'Manager': 2094,
 'Mechanic': 2323,
 'Entrepreneur': 2216,
 'Writer': 2124}

In [ ]:
print(train_data['Customer_ID'].value_counts().head())
print(test_data['Customer_ID'].isin(train_data['Customer_ID']).mean())

Customer_ID
CUS_0xbb77    8
CUS_0x6172    8
CUS_0x5794    8
CUS_0x752c    8
CUS_0x4517    8
Name: count, dtype: int64
0.9764732496323946


In [ ]:
for col in train_data.select_dtypes(exclude='number').columns:
  as_numbers = pd.to_numeric(train_data[col], errors='coerce')
  bad = train_data[col][as_numbers.isna() & train_data[col].notna()]
  print(col, '| not numbers:', len(bad), '|', bad.value_counts().head(5).to_dict())

ID | not numbers: 35879 | {'0x22794': 1, '0x6762': 1, '0x18b64': 1, '0xb8d5': 1, '0x2ec3': 1}
Customer_ID | not numbers: 35879 | {'CUS_0xbb77': 8, 'CUS_0x6172': 8, 'CUS_0x5794': 8, 'CUS_0x752c': 8, 'CUS_0x4517': 8}
Month | not numbers: 35879 | {'January': 4578, 'May': 4497, 'April': 4492, 'August': 4492, 'June': 4487}
Name | not numbers: 35879 | {'N.u': 20, 'Ronald Groverk': 20, 'Stevex': 19, 'Michaelr': 18, 'Nate Raymondw': 18}
Age | not numbers: 1762 | {'24_': 62, '41_': 62, '19_': 62, '38_': 62, '36_': 59}
SSN | not numbers: 35879 | {'#F%$D@*&8': 2032, '344-34-3762': 8, '794-09-7808': 8, '052-90-2364': 8, '483-01-4711': 8}
Occupation | not numbers: 35879 | {'_______': 2547, 'Lawyer': 2370, 'Mechanic': 2323, 'Engineer': 2275, 'Teacher': 2267}
Annual_Income | not numbers: 2430 | {'25856.84_': 3, '15867.45_': 3, '106550.19_': 3, '96583.71_': 3, '17476.37_': 3}
Num_of_Loan | not numbers: 1730 | {'2_': 326, '4_': 294, '3_': 293, '1_': 202, '7_': 185}
Type_of_Loan | not numbers: 35878 | {

# data cleaning

In [ ]:
def OH(data, col, categs):
  data = data.copy()
  for c in categs:
    column = []
    for v in data[col]:
      if v == c:
        column.append(1)
      else:
        column.append(0)
    data[col + '_' + str(c)] = column
  data = data.drop(columns=col)
  return data

In [ ]:
def clean_input(data):
  data = data.copy()

  data = data.drop(columns='Name')
  data = data.drop(columns='SSN')
  data = data.drop(columns='ID')
  data = data.drop(columns='Customer_ID')
  data = data.drop(columns='Type_of_Loan')

  occupations = []
  for v in train_data['Occupation'].dropna().unique():
    if v != '_______':
      occupations.append(v)
  data = OH(data, 'Occupation', occupations)

  payments = []
  for v in train_data['Payment_of_Min_Amount'].dropna().unique():
    payments.append(v)
  data = OH(data, 'Payment_of_Min_Amount', payments)

  payments = []
  for v in train_data['Payment_Behaviour'].dropna().unique():
    payments.append(v)
  data = OH(data, 'Payment_Behaviour', payments)



  names = list(calendar.month_name)
  months = []
  for m in data['Month']:
    months.append(names.index(m))
  data['Month'] = months


  feats = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
         'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly', 'Monthly_Balance']
  for f in feats:
    values = []
    for v in data[f]:
      try:
        values.append(float(str(v).strip('_')))
      except ValueError:
        values.append(np.nan)
    data[f] = values


  mix = []
  for v in data['Credit_Mix']:
    if v == 'Bad':
      mix.append(0)
    elif v == 'Standard':
      mix.append(1)
    elif v == 'Good':
      mix.append(2)
    else:
      mix.append(np.nan)
  data['Credit_Mix'] = mix


  ages = []
  for v in data['Credit_History_Age']:
    if pd.isna(v):
      ages.append(np.nan)
    else:
      parts = v.split(' ')
      ages.append(int(parts[0]) * 12 + int(parts[3]))
  data['Credit_History_Age'] = ages


  return data


In [ ]:
model_data = train_data[train_data['Credit_Score'].notna()].reset_index(drop=True)
y = model_data['Credit_Score'].astype(int)
X = model_data.drop(columns='Credit_Score')

In [ ]:
X_clean = clean_input(X)
test_clean = clean_input(test_data)

print(X_clean.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35878 entries, 0 to 35877
Data columns (total 44 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Month                                               35878 non-null  int64  
 1   Age                                                 35878 non-null  float64
 2   Annual_Income                                       35878 non-null  float64
 3   Monthly_Inhand_Salary                               35878 non-null  float64
 4   Num_Bank_Accounts                                   35878 non-null  int64  
 5   Num_Credit_Card                                     35878 non-null  int64  
 6   Interest_Rate                                       35878 non-null  float64
 7   Num_of_Loan                                         35878 non-null  float64
 8   Delay_from_due_date                                 35878 non-null  float64


# model + validation

In [ ]:
def evaluate(model, X_clean, y):
  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
  scores = []
  for fitrows, valrows in kf.split(X_clean, y):
    X_fit = X_clean.iloc[fitrows]
    y_fit = y.iloc[fitrows]
    X_val = X_clean.iloc[valrows]
    y_val = y.iloc[valrows]

    model.fit(X_fit, y_fit)
    preds = model.predict(X_val)
    score = accuracy_score(y_val, preds)
    scores.append(score)

  print('fold scores:', [round(s, 2) for s in scores])
  print('mean:', round(np.mean(scores), 3), ' spread:', round(np.std(scores), 3))


In [ ]:
evaluate(DummyClassifier(strategy='most_frequent'), X_clean, y)

fold scores: [0.53, 0.53, 0.53, 0.53, 0.53]
mean: 0.528  spread: 0.0


In [ ]:
evaluate(HistGradientBoostingClassifier(random_state=0), X_clean, y)

fold scores: [0.73, 0.72, 0.73, 0.73, 0.73]
mean: 0.732  spread: 0.004


# preds

In [ ]:
model = HistGradientBoostingClassifier(random_state=0)
model.fit(X_clean, y)
preds = model.predict(test_clean)

subtask5 = pd.DataFrame({
    "subtaskID": 5,
    "datapointID": test_data["ID"],
    "answer": preds
})

In [ ]:
submission = pd.concat([subtask1, subtask2, subtask3, subtask4, subtask5], ignore_index=True)
submission.to_csv('output.csv', index=False)